# 1. Chấm Qwen cho submission_17.zip

Upload **hai file dữ liệu** lên `MyDrive/ViBioMIR`: `submission_17.zip` và
`query.parquet` (file local `query/query.parquet`). Mã hỗ trợ được nhúng sẵn trong notebook.
Không cần mở hoặc dán nội dung JSON notebook vào cell Python.

Đây là thử nghiệm trên **120 chunks đã xuất/query**, không phải top 400 cũ.
Giữ baseline ZIP. Qwen không sinh câu trả lời; chỉ chấm mức liên quan query–passage.
Model: Qwen3-Reranker-0.6B, revision 09/06/2025 đã khóa. Không đổi embedding.

Mặc định **INCLUDE_SPANS=False** để thử đổi model trước. Nếu muốn chấm thêm đoạn con,
đặt True trước khi tạo identity; hệ thống sẽ dùng một thư mục run riêng.
Tách câu là heuristic đa ngữ, không phải phân đoạn ngữ nghĩa bằng model.
Giữ chunk gốc làm ứng viên; spans mới luôn là chuỗi con nguyên văn.

Chạy từng cell trong runtime GPU mới. Pilot dừng sau 5 query mới; sau đó chạy một cell
`STOP_AFTER_QUERIES = None` rồi chạy lại đúng cell worker. Không Run all để tiếp tục pilot.
Sau mất kết nối, chạy notebook từ đầu cùng cấu hình; query hoàn tất được xác minh và dùng lại.
Không chạy hai phiên cùng thư mục output. Không sửa hash để dùng cache khác model/input.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, sys, subprocess, time, hashlib, zipfile, threading
ROOT = Path('/content/drive/MyDrive/ViBioMIR')
SUBMISSION_ZIP = ROOT / 'submission_17.zip'
QUERY_FILE = ROOT / 'query.parquet'
SCORE_PARENT = ROOT / 'qwen_submission_trial_v1'
assert SUBMISSION_ZIP.is_file(), f'Upload local submission_17.zip to {SUBMISSION_ZIP}'
assert QUERY_FILE.is_file(), f'Upload local query/query.parquet to {QUERY_FILE}'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')


In [ ]:
# Keep Colab's existing torch installation; dependencies are installed before model import.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow==22.0.0', 'transformers==4.57.1', 'tokenizers==0.22.1',
                'safetensors>=0.4,<1'], check=True)
print('If Colab requests a restart, restart then run setup again before importing models.')


In [ ]:
SUPPORT_SOURCE = '"""Standalone, resumable Qwen scoring of submission evidence and optional subspans.\n\nSource verification here means exact slices of the input submission, not the corpus.\nNo dependency on frozen BGE producer code. No official relevance evaluator included.\n"""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport sqlite3\nimport zipfile\n\nMODEL_ID = \'Qwen/Qwen3-Reranker-0.6B\'\nREVISION = \'6e9e69830b95c52b5fd889b7690dda3329508de3\'\nMODEL_DATE = \'2025-06-09T05:59:29Z\'\nEXPECTED_QUERY_SHA256 = \'ce13508b893bbd6b9169679d540630bc37eb5950cad08b4d5e0e5d7063670c08\'\nEXPECTED_ZIP_SHA256 = \'0963641cb422e4223d6fc4b0c22eb09938f8b03c8227053cc7a398bf33e25dbd\'\nPREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n          \'and the Instruct provided. Note that the answer can only be "yes" or "no".\'\n          \'<|im_end|>\\n<|im_start|>user\\n\')\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\nINSTRUCTION = \'Given a web search query, retrieve relevant passages that answer the query\'\n\n\ndef require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef signature(value):\n    return digest(canonical(value).encode())\n\n\ndef signed(value):\n    return {**value, \'signature\': signature(value)}\n\n\ndef checked(value):\n    require(isinstance(value, dict) and value.get(\'signature\') ==\n            signature({k: v for k, v in value.items() if k != \'signature\'}), \'Record checksum differs\')\n    return value\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(canonical(value), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef generation_policy(include_spans=False):\n    return dict(version=\'sentence-windows-v1\', include_spans=include_spans,\n                min_parent_chars=350, min_span_chars=100, sentence_windows=[1, 2, 3],\n                max_subspans_per_parent=4)\n\n\ndef model_policy():\n    return dict(model_id=MODEL_ID, revision=REVISION, commit_date=MODEL_DATE,\n                instruction=INSTRUCTION, max_pair_tokens=4096, truncation=\'reject\',\n                scoring=\'logit_yes_minus_logit_no\', dtype=\'float16\',\n                attention=\'sdpa\', template_version=\'official-transformers-prefix-suffix-v1\')\n\n\ndef sentence_spans(text):\n    """Conservative boundaries; preserve offsets and avoid splitting decimals/abbreviations on \'.\'."""\n    boundaries = [0]\n    for match in re.finditer(r\'\\n+|[!?。！？]+[ \\t]*\', text):\n        boundaries.append(match.end())\n    # Split full stops only before a likely next sentence. Decimals like 0.10 stay intact.\n    for match in re.finditer(r\'\\.(?:[ \\t]+)(?=[A-ZÀ-Ỹ])\', text):\n        boundaries.append(match.end())\n    boundaries = sorted(set(boundaries + [len(text)]))\n    result = []\n    for a, b in zip(boundaries, boundaries[1:]):\n        while a < b and text[a].isspace():\n            a += 1\n        while b > a and text[b-1].isspace():\n            b -= 1\n        if a < b:\n            result.append((a, b))\n    return result\n\n\ndef evenly_choose(values, count):\n    if len(values) <= count:\n        return values\n    if count == 1:\n        return [values[len(values)//2]]\n    return [values[round(i * (len(values)-1)/(count-1))] for i in range(count)]\n\n\ndef validate_generation(policy):\n    require(policy[\'version\'] == \'sentence-windows-v1\' and type(policy[\'include_spans\']) is bool,\n            \'Unsupported generation policy\')\n    require(all(type(policy[k]) is int and policy[k] > 0 for k in\n                (\'min_parent_chars\', \'min_span_chars\', \'max_subspans_per_parent\')), \'Positive span budgets required\')\n    require(policy[\'sentence_windows\'] and all(type(n) is int and n > 0 for n in policy[\'sentence_windows\']),\n            \'Positive sentence-window sizes required\')\n\n\ndef make_candidates(row, policy):\n    validate_generation(policy)\n    candidates = []\n    for index, parent in enumerate(row[\'relevant_chunks\']):\n        text = parent[\'chunk_text\']\n        spans = [(0, len(text), \'original\')]\n        if policy[\'include_spans\'] and len(text) >= policy[\'min_parent_chars\']:\n            sentences = sentence_spans(text)\n            extras = set()\n            for width in policy[\'sentence_windows\']:\n                for start in range(len(sentences)-width+1):\n                    a, b = sentences[start][0], sentences[start+width-1][1]\n                    if b-a >= policy[\'min_span_chars\'] and b-a < len(text):\n                        extras.add((a, b))\n            spans.extend((a, b, \'subspan\') for a, b in evenly_choose(\n                sorted(extras), policy[\'max_subspans_per_parent\']))\n        parent_hash = digest(text.encode())\n        for a, b, kind in spans:\n            value = dict(parent_index=index, parent_text_sha256=parent_hash,\n                         doc_id=parent[\'doc_id\'], start=a, end=b, kind=kind,\n                         text_sha256=digest(text[a:b].encode()))\n            candidates.append({**value, \'candidate_id\': signature(value)})\n    return candidates\n\n\ndef evidence_text(row, candidate):\n    require(type(candidate[\'parent_index\']) is int and 0 <= candidate[\'parent_index\'] < len(row[\'relevant_chunks\']),\n            \'Unknown parent index\')\n    parent = row[\'relevant_chunks\'][candidate[\'parent_index\']]\n    text = parent[\'chunk_text\']\n    a, b = candidate[\'start\'], candidate[\'end\']\n    require(type(a) is int and type(b) is int and 0 <= a < b <= len(text), \'Invalid evidence offsets\')\n    require(parent[\'doc_id\'] == candidate[\'doc_id\'] and digest(text.encode()) == candidate[\'parent_text_sha256\']\n            and digest(text[a:b].encode()) == candidate[\'text_sha256\'], \'Evidence differs from parent submission\')\n    return text[a:b]\n\n\ndef load_inputs(submission_zip, query_file, expected_zip=EXPECTED_ZIP_SHA256,\n                expected_query=EXPECTED_QUERY_SHA256, expected_queries=1200):\n    require(file_hash(submission_zip) == expected_zip, \'Wrong input ZIP; select the original submission_17.zip\')\n    require(file_hash(query_file) == expected_query, \'Official query file differs\')\n    with zipfile.ZipFile(submission_zip) as z:\n        require(z.namelist() == [\'submission.json\'], \'Expected exactly one root submission.json\')\n        raw = z.read(\'submission.json\')\n    rows = json.loads(raw)\n    import pyarrow.parquet as pq\n    queries = pq.read_table(query_file, columns=[\'id\', \'query\']).to_pylist()\n    validate_input_rows(rows, queries, expected_queries)\n    return rows, queries, dict(submission_zip_sha256=expected_zip, submission_json_sha256=digest(raw),\n                               query_sha256=expected_query), raw\n\n\ndef validate_input_rows(rows, queries, expected_queries):\n    require(isinstance(rows, list) and len(rows) == len(queries) == expected_queries, \'Query count differs\')\n    ids = [q[\'id\'] for q in queries]\n    require(all(type(i) is int for i in ids) and len(set(ids)) == len(ids) and [r[\'id\'] for r in rows] == ids,\n            \'Query IDs/order differ or duplicate IDs\')\n    for row, query in zip(rows, queries):\n        require(isinstance(query[\'query\'], str) and query[\'query\'].strip(), \'Empty query\')\n        docs = row[\'relevant_docs\']\n        require(all(type(d) is int for d in docs) and len(set(docs)) == len(docs), \'Invalid doc list\')\n        for i, c in enumerate(row[\'relevant_chunks\']):\n            require(type(c[\'doc_id\']) is int and c[\'doc_id\'] in docs and isinstance(c[\'chunk_text\'], str)\n                    and c[\'chunk_text\'].strip() and type(c[\'chunk_order\']) is int and c[\'chunk_order\'] == i,\n                    \'Invalid parent evidence\')\n\n\ndef run_identity(inputs, generation, policy):\n    validate_generation(generation)\n    require(policy[\'model_id\'] == MODEL_ID and policy[\'revision\'] == REVISION and policy[\'commit_date\'] == MODEL_DATE,\n            \'Use the pinned Qwen checkpoint\')\n    return dict(operation=\'qwen-submission-rescore-v1\', inputs=inputs, generation=generation,\n                model_policy=policy, code_sha256=file_hash(__file__), source_scope=\'input_submission_only\')\n\n\ndef formatted_input(query, passage, policy):\n    body = f"<Instruct>: {policy[\'instruction\']}\\n<Query>: {query}\\n<Document>: {passage}"\n    return PREFIX + body + SUFFIX\n\n\ndef encoded_pair(tokenizer, query, passage, policy):\n    # Exactly the official prefix/body/suffix assembly, with no silent truncation.\n    body = f"<Instruct>: {policy[\'instruction\']}\\n<Query>: {query}\\n<Document>: {passage}"\n    ids = (tokenizer.encode(PREFIX, add_special_tokens=False)\n           + tokenizer.encode(body, add_special_tokens=False)\n           + tokenizer.encode(SUFFIX, add_special_tokens=False))\n    require(len(ids) <= policy[\'max_pair_tokens\'],\n            f"Pair has {len(ids)} tokens > {policy[\'max_pair_tokens\']}; increase policy budget in a new run")\n    return dict(input_ids=ids, attention_mask=[1]*len(ids))\n\n\nclass QwenScorer:\n    def __init__(self, policy, batch_size=8, padded_tokens=8192):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        require(torch.cuda.is_available(), \'Select a CUDA GPU runtime\')\n        require(policy[\'max_pair_tokens\'] <= 32768 and policy[\'max_pair_tokens\'] > 0\n                and policy[\'truncation\'] == \'reject\' and policy[\'scoring\'] == \'logit_yes_minus_logit_no\'\n                and policy[\'dtype\'] == \'float16\' and policy[\'attention\'] == \'sdpa\', \'Unsupported scoring policy\')\n        require(batch_size > 0 and padded_tokens >= policy[\'max_pair_tokens\'], \'Invalid GPU budgets\')\n        self.torch, self.policy = torch, policy\n        self.device = torch.device(\'cuda\')\n        self.batch_size, self.padded_tokens = batch_size, padded_tokens\n        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION,\n                                                       padding_side=\'left\', trust_remote_code=False)\n        self.model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,\n            torch_dtype=torch.float16, attn_implementation=\'sdpa\', trust_remote_code=False).to(\'cuda\').eval()\n        require(getattr(self.model.config, \'_commit_hash\', None) == REVISION, \'Loaded model revision differs\')\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.encode(\'yes\', add_special_tokens=False)\n        self.no = self.tokenizer.encode(\'no\', add_special_tokens=False)\n        require(len(self.yes) == len(self.no) == 1 and self.yes != self.no, \'Unexpected yes/no tokenizer mapping\')\n        count = sum(p.numel() for p in self.model.parameters())\n        require(0 < count <= 15_000_000_000 and policy[\'max_pair_tokens\'] <= self.model.config.max_position_embeddings,\n                \'Model size/context exceeds allowed budget\')\n        self.registry = dict(model_id=MODEL_ID, revision=REVISION, commit_date=MODEL_DATE,\n                             parameters=count, dtype=\'float16\', scoring=policy[\'scoring\'],\n                             transformers=transformers.__version__, torch=torch.__version__,\n                             tokenizer_revision=REVISION, gpu=torch.cuda.get_device_name(0))\n\n    def score(self, query, passages):\n        features = [encoded_pair(self.tokenizer, query, p, self.policy) for p in passages]\n        scores = []\n        index, budget = 0, self.batch_size\n        while index < len(features):\n            n = min(budget, len(features)-index)\n            while n > 1 and max(len(v[\'input_ids\']) for v in features[index:index+n])*n > self.padded_tokens:\n                n -= 1\n            try:\n                result = self._batch_scores(features[index:index+n])\n            except self.torch.cuda.OutOfMemoryError:\n                self.torch.cuda.empty_cache()\n                if n == 1:\n                    raise\n                budget = max(1, n//2)\n                print(f\'CUDA OOM: retry smaller batch={budget}\', flush=True)\n                continue\n            scores.extend(result)\n            index += n\n        return scores\n\n    def _batch_scores(self, features):\n        inputs = self.tokenizer.pad(features, padding=True, return_tensors=\'pt\')\n        inputs = {k: v.to(self.device) for k, v in inputs.items()}\n        with self.torch.inference_mode():\n            # Keep only the last token logits: avoids full B x sequence x vocabulary allocation.\n            logits = self.model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()\n            values = (logits[:, self.yes[0]] - logits[:, self.no[0]]).cpu().tolist()\n        require(all(math.isfinite(s) for s in values), \'Nonfinite Qwen score\')\n        return values\n\n\ndef cache_key(query, text, policy):\n    return signature(dict(query_sha256=digest(query.encode()), text_sha256=digest(text.encode()), policy=policy))\n\n\ndef verify_score_record(record, row, query, candidates, run_sig, policy):\n    checked(record)\n    require(record[\'id\'] == row[\'id\'] == query[\'id\'] and record[\'run_signature\'] == run_sig and\n            record[\'query_sha256\'] == digest(query[\'query\'].encode()) and\n            record[\'candidates_sha256\'] == signature(candidates), \'Score query/input identity differs\')\n    registry = record[\'model_registry\']\n    require(registry[\'model_id\'] == policy[\'model_id\'] and registry[\'revision\'] == policy[\'revision\'] and\n            registry[\'commit_date\'] == policy[\'commit_date\'] and\n            registry[\'tokenizer_revision\'] == policy[\'revision\'] and registry[\'scoring\'] == policy[\'scoring\'] and\n            registry[\'dtype\'] == policy[\'dtype\'] and type(registry[\'parameters\']) is int and\n            0 < registry[\'parameters\'] <= 15_000_000_000, \'Score model registry differs\')\n    require(len(record[\'scores\']) == len(candidates), \'Missing candidate scores\')\n    for c, score in zip(candidates, record[\'scores\']):\n        require(score[\'candidate_id\'] == c[\'candidate_id\'] and type(score[\'score\']) in (int, float)\n                and math.isfinite(score[\'score\']), \'Invalid score rows\')\n    return [v[\'score\'] for v in record[\'scores\']]\n\n\ndef run_scoring(rows, queries, identity, output, local_cache, *, stop_after_queries=5,\n                batch_size=8, padded_tokens=8192, scorer_factory=None):\n    require(stop_after_queries is None or type(stop_after_queries) is int and stop_after_queries > 0,\n            \'Positive pilot limit or None required\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_path = output / \'manifest.json\'\n    run_sig = signature(identity)\n    if manifest_path.exists():\n        previous = checked(read(manifest_path))\n        require(previous[\'identity\'] == identity, \'Scoring inputs changed; use new run directory\')\n    else:\n        require(not any(output.iterdir()), \'Nonempty run without manifest; use new directory\')\n    done, hashes, new_count, scored_pairs = [], {}, 0, 0\n    scorer = None\n    def save(state):\n        write(manifest_path, signed(dict(identity=identity, run_signature=run_sig, state=state,\n            completed_query_ids=done, score_files=hashes, queries=len(rows),\n            new_queries_last_invocation=new_count, scored_unique_pairs_last_invocation=scored_pairs)))\n    save(\'running\')\n    Path(local_cache).parent.mkdir(parents=True, exist_ok=True)\n    with sqlite3.connect(local_cache) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,score REAL NOT NULL)\')\n        try:\n            for row, query in zip(rows, queries):\n                candidates = make_candidates(row, identity[\'generation\'])\n                path = output / f"query-{row[\'id\']}.json"\n                if path.is_file():\n                    verify_score_record(read(path), row, query, candidates, run_sig, identity[\'model_policy\'])\n                else:\n                    keys, texts, missing = [], {}, []\n                    for c in candidates:\n                        text = evidence_text(row, c)\n                        key = cache_key(query[\'query\'], text, identity[\'model_policy\'])\n                        keys.append(key)\n                        texts[key] = text\n                    values = {}\n                    for key in dict.fromkeys(keys):\n                        hit = db.execute(\'SELECT score FROM scores WHERE key=?\', (key,)).fetchone()\n                        if hit:\n                            require(math.isfinite(hit[0]), \'Nonfinite cached score\')\n                            values[key] = hit[0]\n                        else:\n                            missing.append(key)\n                    if scorer is None:\n                        scorer = (scorer_factory or QwenScorer)(identity[\'model_policy\'],\n                            batch_size=batch_size, padded_tokens=padded_tokens)\n                    if missing:\n                        scores = scorer.score(query[\'query\'], [texts[k] for k in missing])\n                        require(len(scores) == len(missing) and all(type(s) in (int, float) and math.isfinite(s) for s in scores),\n                                \'Scorer returned invalid scores\')\n                        for key, score in zip(missing, scores):\n                            db.execute(\'INSERT INTO scores VALUES(?,?)\', (key, float(score)))\n                            values[key] = float(score)\n                        db.commit()\n                        scored_pairs += len(missing)\n                    record = signed(dict(id=row[\'id\'], run_signature=run_sig,\n                        query_sha256=digest(query[\'query\'].encode()), candidates_sha256=signature(candidates),\n                        model_registry=scorer.registry,\n                        scores=[dict(candidate_id=c[\'candidate_id\'], score=values[k]) for c,k in zip(candidates,keys)]))\n                    verify_score_record(record, row, query, candidates, run_sig, identity[\'model_policy\'])\n                    write(path, record)\n                    new_count += 1\n                done.append(row[\'id\']); hashes[str(row[\'id\'])] = file_hash(path)\n                save(\'running\')\n                print(f"Query {row[\'id\']}: {len(done)}/{len(rows)} complete; new this call={new_count}", flush=True)\n                if stop_after_queries is not None and new_count >= stop_after_queries and len(done) < len(rows):\n                    save(\'paused\')\n                    return read(manifest_path)\n        except BaseException:\n            save(\'interrupted\')\n            raise\n    save(\'complete\')\n    return read(manifest_path)\n\n\ndef overlap_iou(a, b):\n    if (a[\'doc_id\'], a[\'parent_index\']) != (b[\'doc_id\'], b[\'parent_index\']):\n        return 0.0  # Source coordinates across different parent chunks are unavailable.\n    overlap = max(0, min(a[\'end\'], b[\'end\']) - max(a[\'start\'], b[\'start\']))\n    return overlap / (max(a[\'end\'], b[\'end\']) - min(a[\'start\'], b[\'start\']))\n\n\ndef select_evidence(row, candidates, scores, limits, originals_only=False):\n    require(type(limits[\'max_chunks\']) is int and limits[\'max_chunks\'] > 0 and\n            type(limits[\'per_doc\']) is int and limits[\'per_doc\'] > 0 and\n            0 < limits[\'overlap_iou\'] <= 1, \'Invalid selection limits\')\n    require(len(candidates) == len(scores) and all(math.isfinite(s) for s in scores), \'Invalid score array\')\n    selected, per_doc, decisions, texts = [], {}, [], {}\n    ranked = sorted(zip(candidates, scores), key=lambda x: (-x[1], x[0][\'candidate_id\']))\n    for c, score in ranked:\n        text = evidence_text(row, c)\n        reason = None\n        if originals_only and c[\'kind\'] != \'original\':\n            reason = \'subspan_disabled\'\n        elif len(selected) >= limits[\'max_chunks\']:\n            reason = \'global_cap\'\n        elif per_doc.get(c[\'doc_id\'], 0) >= limits[\'per_doc\']:\n            reason = \'per_doc_cap\'\n        else:\n            peers = [v for v in selected if v[\'doc_id\'] == c[\'doc_id\']]\n            if any(text == texts[v[\'candidate_id\']] for v in peers):\n                reason = \'same_doc_exact_duplicate\'\n            elif any(overlap_iou(c, v) >= limits[\'overlap_iou\'] for v in peers):\n                reason = \'same_parent_overlap\'\n        if reason is None:\n            selected.append(c)\n            texts[c[\'candidate_id\']] = text\n            per_doc[c[\'doc_id\']] = per_doc.get(c[\'doc_id\'], 0)+1\n        decisions.append(dict(candidate_id=c[\'candidate_id\'], reason=reason or \'selected\', score=score))\n    prediction = dict(id=row[\'id\'], relevant_docs=list(row[\'relevant_docs\']), relevant_chunks=[\n        dict(doc_id=c[\'doc_id\'], chunk_text=texts[c[\'candidate_id\']], chunk_order=i) for i,c in enumerate(selected)])\n    return prediction, selected, decisions\n\n\ndef package_variants(rows, queries, identity, score_dir, destination, baseline_raw, variants):\n    score_dir, destination = Path(score_dir), Path(destination)\n    require(not destination.exists(), \'Choose a new export directory\')\n    require(digest(baseline_raw) == identity[\'inputs\'][\'submission_json_sha256\'], \'Baseline bytes differ\')\n    manifest = checked(read(score_dir / \'manifest.json\'))\n    require(manifest[\'identity\'] == identity and manifest[\'state\'] == \'complete\' and\n            manifest[\'completed_query_ids\'] == [r[\'id\'] for r in rows], \'Finish all scoring queries before export\')\n    run_sig = signature(identity)\n    manifest_hash = file_hash(score_dir / \'manifest.json\')\n    destination.mkdir(parents=True)\n    # S0 is byte-for-byte baseline JSON, repackaged without changing prediction data.\n    with zipfile.ZipFile(destination / \'S0_baseline.zip\', \'w\', zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'submission.json\', baseline_raw)\n    predictions, counts = {v[\'name\']: [] for v in variants}, {v[\'name\']: [] for v in variants}\n    require(len(predictions) == len(variants) and all(re.fullmatch(r\'[A-Za-z0-9_-]+\', v[\'name\']) for v in variants),\n            \'Unique safe variant names required\')\n    evidence_sidecar = destination / \'EVIDENCE_PROVENANCE.jsonl\'\n    with evidence_sidecar.open(\'w\', encoding=\'utf-8\') as sidecar:\n        for row, query in zip(rows, queries):\n            candidates = make_candidates(row, identity[\'generation\'])\n            path = score_dir / f"query-{row[\'id\']}.json"\n            require(file_hash(path) == manifest[\'score_files\'][str(row[\'id\'])], \'Score file changed after completion\')\n            scores = verify_score_record(read(path), row, query, candidates, run_sig, identity[\'model_policy\'])\n            for variant in variants:\n                pred, selected, decisions = select_evidence(row, candidates, scores, variant[\'limits\'],\n                                                           variant[\'originals_only\'])\n                require(pred[\'relevant_docs\'] == row[\'relevant_docs\'], \'Document list changed\')\n                predictions[variant[\'name\']].append(pred)\n                before = {(c[\'doc_id\'], c[\'chunk_text\']) for c in row[\'relevant_chunks\']}\n                after = {(c[\'doc_id\'], c[\'chunk_text\']) for c in pred[\'relevant_chunks\']}\n                counts[variant[\'name\']].append(dict(id=row[\'id\'], chunks=len(selected),\n                    changed_evidence=before != after, subspans=sum(c[\'kind\']==\'subspan\' for c in selected),\n                    documents_with_chunks=len({c[\'doc_id\'] for c in selected})))\n                sidecar.write(canonical(dict(variant=variant[\'name\'], id=row[\'id\'], selected=selected,\n                                             decisions=decisions))+\'\\n\')\n    require(file_hash(score_dir / \'manifest.json\') == manifest_hash, \'Scoring manifest changed during export\')\n    exports = []\n    for variant in variants:\n        name = variant[\'name\']\n        payload = canonical(predictions[name]).encode()\n        archive = destination / f\'{name}.zip\'\n        with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'submission.json\', payload)\n        with zipfile.ZipFile(archive) as z:\n            require(z.namelist() == [\'submission.json\'] and z.read(\'submission.json\') == payload, \'ZIP differs\')\n        exports.append(dict(variant=variant, zip=archive.name, zip_sha256=file_hash(archive),\n            json_sha256=digest(payload), queries=len(rows),\n            changed_queries=sum(c[\'changed_evidence\'] for c in counts[name]),\n            total_chunks=sum(c[\'chunks\'] for c in counts[name]),\n            total_subspans=sum(c[\'subspans\'] for c in counts[name]), query_statistics=counts[name]))\n    report = dict(validation=\'passed\', validation_scope=\'input submission subset/slices and score integrity\',\n        original_corpus_evidence_verified=False, official_doc_allowlist_verified=False,\n        relevance_measured=False, official_lcs_evaluated=False, source_identity=identity,\n        score_manifest_sha256=manifest_hash, baseline_json_sha256=digest(baseline_raw),\n        provenance_sha256=file_hash(evidence_sidecar), exports=exports)\n    write(destination / \'REPORT.json\', report)\n    return report\n'
SUPPORT_DIR = Path('/content/qwen_submission_trial_code')
SUPPORT_DIR.mkdir(parents=True, exist_ok=True)
(SUPPORT_DIR / 'qwen_submission_trial_support.py').write_text(SUPPORT_SOURCE, encoding='utf-8')
assert 'qwen_submission_trial_support' not in sys.modules, 'Helper already loaded: continue from input cell or restart runtime before loading different code.'
sys.path.insert(0, str(SUPPORT_DIR))
from qwen_submission_trial_support import (
    MODEL_ID, REVISION, MODEL_DATE, load_inputs, generation_policy, model_policy,
    run_identity, signature, signed, checked, read, write, make_candidates,
    run_scoring, package_variants,
)
from contextlib import contextmanager
@contextmanager
def progress(label):
    event = threading.Event()
    started = time.monotonic()
    def heartbeat():
        while not event.wait(30):
            print(label, 'minutes:', round((time.monotonic()-started)/60, 1), flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    print(label, flush=True)
    thread.start()
    try:
        yield
    finally:
        event.set()
        thread.join()


In [ ]:
with progress('Read and validate ZIP + official query snapshot'):
    rows, queries, inputs, baseline_raw = load_inputs(SUBMISSION_ZIP, QUERY_FILE)
print('Queries:', len(rows), 'Original chunks:', sum(len(r['relevant_chunks']) for r in rows))
print('Verified input hashes:', inputs)
print('Source scope: exact slices of the submitted chunks; original corpus is not loaded.')


In [ ]:
INCLUDE_SPANS = False  # True adds up to 4 subspans per long original chunk.
GENERATION = generation_policy(INCLUDE_SPANS)
POLICY = model_policy()
BATCH_SIZE = 8
PADDED_TOKENS = 8192
STOP_AFTER_QUERIES = 5
identity = run_identity(inputs, GENERATION, POLICY)
RUN = SCORE_PARENT / signature(identity)
LOCAL_CACHE = Path('/content/qwen_submission_trial_cache') / (signature(identity) + '.sqlite3')
with progress('Count original and optional subspan pairs'):
    sizes = [len(make_candidates(row, GENERATION)) for row in rows]
print('Model:', MODEL_ID, 'revision:', REVISION, 'date:', MODEL_DATE)
print('Pairs before text dedup:', sum(sizes), 'max/query:', max(sizes))
print('Output directory:', RUN)
print('This does not measure relevance or official LCS score.')


In [ ]:
# To finish after pilot: run STOP_AFTER_QUERIES=None in a separate cell, then rerun THIS cell.
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))
started = time.monotonic()
with progress('Qwen scoring'):
    result = run_scoring(rows, queries, identity, RUN, LOCAL_CACHE,
        stop_after_queries=STOP_AFTER_QUERIES, batch_size=BATCH_SIZE, padded_tokens=PADDED_TOKENS)
print('State:', result['state'])
print('Verified completed queries:', len(result['completed_query_ids']), '/', len(rows))
print('New queries this call:', result['new_queries_last_invocation'])
print('Seconds:', round(time.monotonic()-started, 1))
if result['state'] != 'complete':
    print('PILOT/PARTIAL: set STOP_AFTER_QUERIES=None, rerun only this worker cell. Do not export yet.')
else:
    print('COMPLETE: open stage9_qwen_select_submission.ipynb (CPU is sufficient).')
    print('Use RUN_DIRECTORY = Path(' + repr(str(RUN)) + ')')
